In [0]:
import pyspark.sql.functions as F

In [0]:
# Rename columns using medata_dataframe
def renameCols(metadata_df, dataframe, filename):
    for row in metadata_df.filter(F.col("SourceFileName")== file_name).rdd.collect():
        dataframe = dataframe.withColumnRenamed(
            row.SourceFieldName,
            row.CleansedFieldName
        )
        return dataframe


In [0]:
file_name ="SampleDataSuperStore.csv"
container = "bronze"

path = f'abfss://{container}@{account_name}.dfs.core.windows.net/{file_name}'

df = spark.read.csv(path, header="true", inferSchema="true", sep=',')

df.display()



Row ID,Order ID,Order Date,Ship Date,Customer Name,Segment,Country,Postal Code,Sales,Quantity,Discount,Profit
1,CA-2013-152156,2013-11-09,2013-11-12,Claire Gute,Consumer,United States,42420,261.96,2,0.0,41.9136
2,CA-2013-152156,2013-11-09,2013-11-12,Claire Gute,Consumer,United States,42420,731.94,3,0.0,219.582
3,CA-2013-138688,2013-06-13,2013-06-17,Darrin Van Huff,Corporate,United States,90036,14.62,2,0.0,6.8714
4,US-2012-108966,2012-10-11,2012-10-18,Sean O'Donnell,Consumer,United States,33311,957.5775,5,0.45,-383.031
5,US-2012-108966,2012-10-11,2012-10-18,Sean O'Donnell,Consumer,United States,33311,22.368,2,0.2,2.5164
6,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,Consumer,United States,90032,48.86,7,0.0,14.1694
7,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,Consumer,United States,90032,7.28,4,0.0,1.9656
8,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,Consumer,United States,90032,907.152,6,0.2,90.7152
9,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,Consumer,United States,90032,18.504,3,0.2,5.7825
10,CA-2011-115812,2011-06-09,2011-06-14,Brosina Hoffman,Consumer,United States,90032,114.9,5,0.0,34.47


In [0]:
df_hash = df.withColumn(
    "Customer_Name_Hash", F.hash(F.col("Customer Name"))
).withColumn(
    "Zip_Code_Hash", F.hash(F.col("Postal Code"))
).drop(*["Customer Name", "Postal Code"])
df_hash.display()

Row ID,Order ID,Order Date,Ship Date,Segment,Country,Sales,Quantity,Discount,Profit,Customer_Name_Hash,Zip_Code_Hash
1,CA-2013-152156,2013-11-09,2013-11-12,Consumer,United States,261.96,2,0.0,41.9136,1955614814,928416183
2,CA-2013-152156,2013-11-09,2013-11-12,Consumer,United States,731.94,3,0.0,219.582,1955614814,928416183
3,CA-2013-138688,2013-06-13,2013-06-17,Corporate,United States,14.62,2,0.0,6.8714,793812325,1007571151
4,US-2012-108966,2012-10-11,2012-10-18,Consumer,United States,957.5775,5,0.45,-383.031,526017428,-1006865958
5,US-2012-108966,2012-10-11,2012-10-18,Consumer,United States,22.368,2,0.2,2.5164,526017428,-1006865958
6,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,48.86,7,0.0,14.1694,-1753018072,1808393599
7,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,7.28,4,0.0,1.9656,-1753018072,1808393599
8,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,907.152,6,0.2,90.7152,-1753018072,1808393599
9,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,18.504,3,0.2,5.7825,-1753018072,1808393599
10,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,114.9,5,0.0,34.47,-1753018072,1808393599


In [0]:
df_renamed_cols = df_hash.withColumnRenamed("Order ID", "order_id")
df_renamed_cols.display()

Row ID,order_id,Order Date,Ship Date,Segment,Country,Sales,Quantity,Discount,Profit,Customer_Name_Hash,Zip_Code_Hash
1,CA-2013-152156,2013-11-09,2013-11-12,Consumer,United States,261.96,2,0.0,41.9136,1955614814,928416183
2,CA-2013-152156,2013-11-09,2013-11-12,Consumer,United States,731.94,3,0.0,219.582,1955614814,928416183
3,CA-2013-138688,2013-06-13,2013-06-17,Corporate,United States,14.62,2,0.0,6.8714,793812325,1007571151
4,US-2012-108966,2012-10-11,2012-10-18,Consumer,United States,957.5775,5,0.45,-383.031,526017428,-1006865958
5,US-2012-108966,2012-10-11,2012-10-18,Consumer,United States,22.368,2,0.2,2.5164,526017428,-1006865958
6,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,48.86,7,0.0,14.1694,-1753018072,1808393599
7,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,7.28,4,0.0,1.9656,-1753018072,1808393599
8,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,907.152,6,0.2,90.7152,-1753018072,1808393599
9,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,18.504,3,0.2,5.7825,-1753018072,1808393599
10,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,114.9,5,0.0,34.47,-1753018072,1808393599


In [0]:
df_renamed_cols = df_hash.withColumnRenamed(
    "Order ID", "order_id"
    ).withColumnRenamed(
        "Order Date", "order_date"
        ).withColumnRenamed(
        "Ship Date", "ship_date"
        )
df_renamed_cols.display()

Row ID,order_id,order_date,ship_date,Segment,Country,Sales,Quantity,Discount,Profit,Customer_Name_Hash,Zip_Code_Hash
1,CA-2013-152156,2013-11-09,2013-11-12,Consumer,United States,261.96,2,0.0,41.9136,1955614814,928416183
2,CA-2013-152156,2013-11-09,2013-11-12,Consumer,United States,731.94,3,0.0,219.582,1955614814,928416183
3,CA-2013-138688,2013-06-13,2013-06-17,Corporate,United States,14.62,2,0.0,6.8714,793812325,1007571151
4,US-2012-108966,2012-10-11,2012-10-18,Consumer,United States,957.5775,5,0.45,-383.031,526017428,-1006865958
5,US-2012-108966,2012-10-11,2012-10-18,Consumer,United States,22.368,2,0.2,2.5164,526017428,-1006865958
6,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,48.86,7,0.0,14.1694,-1753018072,1808393599
7,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,7.28,4,0.0,1.9656,-1753018072,1808393599
8,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,907.152,6,0.2,90.7152,-1753018072,1808393599
9,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,18.504,3,0.2,5.7825,-1753018072,1808393599
10,CA-2011-115812,2011-06-09,2011-06-14,Consumer,United States,114.9,5,0.0,34.47,-1753018072,1808393599


In [0]:
file_name2 ="SampleData - TSS_EVENT_.csv"

path2 = f'abfss://{container}@{account_name}.dfs.core.windows.net/{file_name2}'

df2 = spark.read.csv(path2, header="false", inferSchema="true", sep=',')

df2.display()

_c0
BNSF 65319007PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 3449-02-23-20.59.19.302019002APPL X
BNSF 65320007PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 5328-02-23-20.59.19.232019002APPL X
BNSF 65321307PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 2876-02-23-20.59.18.432019002APPL X
BNSF 65325907PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 4438-02-23-20.59.18.862019002APPL X
BNSF 65325607PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 7867-02-23-20.59.18.492019002APPL X
BNSF 65335407PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 4423-02-23-20.59.19.392019002APPL X
BNSF 65337507PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 6214-02-23-20.59.18.792019002APPL X
BNSF 65338007PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 1596-02-23-20.59.18.372019002APPL X
BNSF 65337707PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 4543-02-23-20.59.18.412019002APPL X
BNSF 65340207PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 0787-02-23-20.59.18.442019002APPL X


In [0]:
df2_substring = df2.withColumn(
    "equipment_initial", F.substring(F.col("_c0"), 1,4)
    ).withColumn(
        "equipment_number", F.substring(F.col("_c0"), 5,14))
df2_substring.display()

_c0,equipment_initial,equipment_number
BNSF 65319007PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 3449-02-23-20.59.19.302019002APPL X,BNSF,65319007PA
BNSF 65320007PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 5328-02-23-20.59.19.232019002APPL X,BNSF,65320007PA
BNSF 65321307PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 2876-02-23-20.59.18.432019002APPL X,BNSF,65321307PA
BNSF 65325907PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 4438-02-23-20.59.18.862019002APPL X,BNSF,65325907PA
BNSF 65325607PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 7867-02-23-20.59.18.492019002APPL X,BNSF,65325607PA
BNSF 65335407PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 4423-02-23-20.59.19.392019002APPL X,BNSF,65335407PA
BNSF 65337507PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 6214-02-23-20.59.18.792019002APPL X,BNSF,65337507PA
BNSF 65338007PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 1596-02-23-20.59.18.372019002APPL X,BNSF,65338007PA
BNSF 65337707PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 4543-02-23-20.59.18.412019002APPL X,BNSF,65337707PA
BNSF 65340207PACT 2019-02-2710.40.0033158 EE0PHPNAM03BNSF 0787-02-23-20.59.18.442019002APPL X,BNSF,65340207PA


In [0]:
# Rename columns using medata_dataframe
def renameCols(metadata_df, dataframe, filename):
    for row in metadata_df.filter(F.col("SourceFileName")== file_name).rdd.collect():
        dataframe = dataframe.withColumnRenamed(
            row.SourceFieldName,
            row.CleansedFieldName
        )
        return dataframe


In [0]:
file_name3 ="MetaData.csv"
container = "dropzone"

path3 = f'abfss://{container}@{account_name}.dfs.core.windows.net/{file_name3}'

metadata_df = spark.read.csv(path3, header="true", inferSchema="true", sep=',')

metadata_df.display()

SourceName,SourceFileName,SourceFieldName,CleansedFileName,CleansedFieldName,CleansedFieldType,Start Position,End Position,File Type,PII,_c10,_c11,_c12,_c13,_c14,_c15
Database_SampleData,SampleData - SuperStoreXref,Customer Return,sampledata - superstorexref,customer Return,string,null,null,csv,No,null,null,null,null,null,null
Database_SampleData,SampleData - SuperStoreXref,Order ID,sampledata - superstorexref,order id,string,null,null,csv,No,null,null,null,null,null,null
API,SampleData - TSS_EVENT_,null,sampledata - tss_event_,equipment_initial,string,1,4,csv,No,null,null,null,null,null,null
API,SampleData - TSS_EVENT_,null,sampledata - tss_event_,equipment_number,integer,5,14,csv,No,null,null,null,null,null,null
API,SampleData - TSS_EVENT_,null,sampledata - tss_event_,event_code,integer,15,16,csv,No,null,null,null,null,null,null
API,SampleData - TSS_EVENT_,null,sampledata - tss_event_,event_code_alpha,string,17,20,csv,No,null,null,null,null,null,null
API,SampleData - TSS_EVENT_,null,sampledata - tss_event_,cancel_date,date,21,30,csv,No,null,null,null,null,null,null
API,SampleData - TSS_EVENT_,null,sampledata - tss_event_,event_date,datetime,31,48,csv,No,null,null,null,null,null,null
API,SampleData - TSS_EVENT_,null,sampledata - tss_event_,fsac,integer,49,54,csv,No,null,null,null,null,null,null
API,SampleData - TSS_EVENT_,null,sampledata - tss_event_,load_empty,string,55,55,csv,No,null,null,null,null,null,null


In [0]:
for row in metadata_df.filter(F.col("SourceFileName")== file_name).rdd.collect():
        dataframe = dataframe.withColumn(
            row.SourceFieldName,
            row.CleansedFieldName
        )
        return dataframe

  File <command-1691068207814548>:6
    return dataframe
    ^
SyntaxError: 'return' outside function


In [0]:
df_final = renameCols(metadata_df, dataframe, "SampleDataSuperStore" )
df_final.display()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-1691068207814545>:1
----> 1 df_final = renameCols(metadata_df, dataframe, "SampleDataSuperStore" )
      2 df_final.display()

NameError: name 'dataframe' is not defined